## **Output Parsers**

In [4]:
# Output Parser - A simple tool to parse and format output from various commands.
# we are having a problem with the output of the command, we want to parse it and format it in a more readable way.
# we have so many parsers available in langchain, we can use them to parse the output of the command and format it in a more readable way.
# here we are going to use few of the important and usable parsers as:

# 1. StrOutputParser - to parse the output in string format and format it in a more readable way.
    # this parse the output in simple string format.
# 2. JsonOutputParser - to parse the output in JSON format and format it in a more readable way.
    # this parse the output in JSON format but we can't explicitly define the schema of the output, it will just parse the output in JSON format by default schema.
# 3. StructuredOutputParser - to parse the output in JSON format and format it in a more readable way.
    # this parse the output in JSON format and we can explicitly define the schema of the output, it will parse the output in JSON format by default schema.
    # but we can't get validation error if the output is not in the defined schema, this pushes the use of PydanticOutputParser.
# 4. PydanticOutputParser - to parse the output in JSON format and format it in a more readable way.
    # this parse the output in JSON format and we can explicitly define the schema of the output, it will parse the output in JSON format by default schema.
    # we can get validation error if the output is not in the defined schema, this is the best parser to use when we want to parse the output in JSON format and we want to validate the output against a defined schema.

# few models doesn't provide inbuilt support for structured output (withstructure output), for them we need to use the output parsers (OutputFixingParser), this parser will fix the output of the command and make it in a structured format that can be parsed by the other parsers.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",max_tokens=2048)

**StrOutputParser**

In [6]:

from langchain_core.output_parsers import StrOutputParser

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

parser = StrOutputParser()

chain = template1 | model | parser | template2 | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

Black holes are extreme cosmic regions where gravity is so intense that nothing, not even light, can escape. Predicted by Einstein, they are incredibly dense concentrations of matter, fundamentally altering spacetime. Their immense mass is compressed into a tiny volume, causing the escape velocity to exceed the speed of light. The Event Horizon is the critical boundary beyond which matter and light are irrevocably trapped. Studying them, along with their core singularity, offers profound insights into gravity and the universe.


**JSON Output Parser**

In [9]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

template = PromptTemplate(
    template='Give me 3 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})
print(result)

{'facts': ['Black holes are regions in spacetime where gravity is so strong that nothing, not even light, can escape from them.', 'Most known black holes form from the gravitational collapse of massive stars at the end of their life cycle.', "The boundary around a black hole beyond which no escape is possible is called the event horizon, often referred to as the 'point of no return'."]}


**StructuredOutputParser**

In [2]:
from langchain_classic.output_parsers import StructuredOutputParser

#not available in langchain_core, only in langchain, or it depricated.

**Pydantic Output Parser**

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Person(BaseModel):

    name: str = Field(description='Name of the person')
    age: int = Field(gt=18, description='Age of the person')
    city: str = Field(description='Name of the city the person belongs to')


parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template = 'generate the name, age and city of a person cosidering the person at place: {place}. \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'place':'Chhattisgarh,India'})

print(result)

name='Priya Sharma' age=28 city='Raipur'
